# Data Science in Psychology and Neuroscience

## Class info:
* Week #11
* Day: March 31, 2026
* Time: 9:30—10:45 AM
* Location: Logan Hall 125
* <a href="https://forms.microsoft.com/r/26vAcJWrwH">Click here to submit your attendance for Week 11, Question 19!</a>
  
## Instructor info:
* Dr. Jeremy Hogeveen
* jhogeveen@unm.edu
* Logan Hall 281 (Office Hours By Appointment)
  
## Syllabus:
* <a href="https://www.dropbox.com/scl/fi/6fs6fi4kvkwtxn7j7x8ua/PSY450650_DSPN_Spring2026_Syllabus.pdf?rlkey=148e5t4ah8q2n1daclt7mp0h6&dl=0">Download here</a>

## Today's topic:
* Modeling our ketamine trial data, continued
1. Repeated-measures ANOVA
2. Three by three mixed ANOVA (the omnibus test)
3. When ANOVAs let you down...
4. Linear Mixed Models (LMMs).
    * Fit our first LMM using `lme4`.
 

# Section 1. Repeated-Measures ANOVA, continuation from last class

## 1.1 Are there differences between the pre, post, _and_ followup HAMD changes in the ketamine group.

In [1]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(rstatix))
suppressPackageStartupMessages(library(afex))
suppressPackageStartupMessages(library(emmeans))
suppressPackageStartupMessages(library(knitr))

df <- read_csv('../data/synth_ketamine_data.csv')


# Reorder the factor levels in your data frame
desired_order <- c("Pre", "Post", "Followup")
df$Time <- factor(df$Time, levels = desired_order)

head(df)

Rows: 180 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): Drug, Time
dbl (3): Subject, Change, HAMD

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Subject,Drug,Change,Time,HAMD
<dbl>,<chr>,<dbl>,<fct>,<dbl>
1,Ketamine,10.89830,Pre,28.112875
1,Ketamine,10.89830,Post,17.214579
1,Ketamine,10.89830,Followup,15.227329
2,Ketamine,12.55569,Pre,22.305905
2,Ketamine,12.55569,Post,9.750214
2,Ketamine,12.55569,Followup,7.809342


* Modeling our ketamine trial data, continued
1. ~~Repeated-measures ANOVA~~
2. Three by three mixed ANOVA (the omnibus test)
3. When ANOVAs let you down...
4. Linear Mixed Models (LMMs).
    * Fit our first LMM using `lme4`.

# Section 2. The omnibus ANOVA

## 2.1 Running a `3 (time: pre, posttest, followup)` x `3 (drug: ketamine, amphetamine, placebo)` Mixed ANOVA
*  Are there differences in the long-term antidepressant effects of ketamine versus amphetamine versus placebo?
*  This represents the omnibus test for this design.
    *  AKA, what we'd actually run first if this was our study. Why?
        * Multiple comparisons issues:
            * At this point, we'd run i) a `paired t-test`, ii) an `independent t-test`, iii) a `one-way ANOVA`, and iv) a `rm-ANOVA`.
            * Family-Wise Error Rate (FWER): $FWER =  1 - (1 - \alpha)^n$.
                * $FWER = 1 - (1 - 0.95)^4$
                * $FWER = 0.1855$
                * False positive rate has ≈quadrupled.
        * Maximizes statistical power:
            * Using all subjects + data points to estimate error variance.
            * Higher error degrees of freedom in the ANOVA denominator, lower critical threshold for significance.
        * Capturing the critical `interaction term` (drug * time)
            * Does the trajectory of change depend on the drug? Only the interaction term in the omnibus model does that.

* Modeling our ketamine trial data, continued
1. ~~Repeated-measures ANOVA~~
2. ~~Three by three mixed ANOVA (the omnibus test)~~
3. When ANOVAs let you down...
4. Linear Mixed Models (LMMs).
    * Fit our first LMM using `lme4`.

# Section 3. When ANOVAs let us down.
### 3.1 What if ≈25% of our subjects did not return at 6-month follow-up?

## 3.2 Key scenarios when linear mixed models (LMMs) are superior to ANOVA
* Missing data.
* Unbalanced designs.
* Autocorrelated data.
    * Timeseries data.
    * Longitudinal data.
* Complex nested data structures.
    * e.g. Students within classrooms, patients within diagnostic categories, animals within a litter, etc.
* Continuous time.
* Strong violations of sphericity assumptions in rmANOVA.

### Modeling our ketamine trial data, continued
1. ~~Repeated-measures ANOVA~~
2. ~~Three by three mixed ANOVA (the omnibus test)~~
3. ~~When ANOVAs let you down...~~
4. Linear Mixed Models (LMMs).
    * Fit our first LMM using `lme4`.

# Section 4. Linear Mixed Models (LMMs)
* AKA mixed models, multilevel models, hierarchical models.
* LMMs don't rely on perfect, balanced matrices.
    * They look at all available data points.
    * If a subject has a Pre and Post score but no Followup, the LMM still uses their Pre and Post data to estimate the overall trajectories!
    * Model works by maximizing the likelihood of the model given _all_ the data we have.

## 4.1 But how?

* To understand how Linear Mixed Models handle this data, we need to split our variables into two distinct categories: `Fixed Effects` and `Random Effects`.
* Fixed Effects (`Drug` and `Time`): These are the experimental variables we actually care about testing and generalizing to the broader population. We want to know the *universal, average* effect of Ketamine versus Placebo over time. 
* Random Effects (`Subject`): This accounts for the idiosyncratic "noise" in our specific sample. We don't care about Subject #42's specific life story, but we *do* know that Subject #42 might naturally start with higher depression scores than Subject #12. Random effects allow us to mathematically control for this individual variation.

* The Syntax: `HAMD ~ Drug * Time + (1 | Subject)`
1. `Drug * Time` represents our ___Fixed Effects___. 
2. `(1 | Subject)` tells the model to add a ___Random Intercept___ for each person.

* In plain English: _"Calculate the overall, average effects of the Drug over Time, but allow every single subject to start at their own unique baseline level of depression."_

## 4.2 Running the LMM and Checking Assumptions
* Now that we understand the difference between fixed and random effects, let's fit our model and look at the residuals.
    * Just like with standard ANOVAs, we want to make sure our errors are normally distributed and roughly equal across our groups.

## 4.3 Visualizing the LMM interaction
* To plot our interaction, we need to extract the predicted values from our model. 
    * Here is the critical difference between plotting a standard `lm()` and an `lmer()` model: If we just ask the model for predictions, it will try to draw a unique line for every single subject based on their Random Intercept!
    * Since we want to plot the overall, universal effect of the drugs, we need to tell the `predict()` function to ignore the random subject variance and only calculate the ___Fixed Effects___. We do this using the argument `re.form = NA`.

## 4.4 The flexibility of LMMs
* LMMs can handle designs much more sophisticated than our simple `(1 | Subject)` random intercept.
* Two incredibly common extensions in clinical and neuroscience research are:
    * Nested Designs:
        * What if our sample included patients with two different underlying conditions (e.g., Major Depressive Disorder vs. Bipolar Depression)?
        * We can model this explicitly as a ___nested random intercept___
            * Changing our syntax to `(1 | Diagnosis/Subject)` tells the model: _"Subjects belong to specific Diagnostic groups; calculate an overarching baseline for MDD vs. BPD, and then calculate each individual Subject's baseline within their respective group."_
    * Random Slopes:
      * Our model assumes the drug worked at the exact same _rate_ for everyone (i.e., random intercepts, parallel slopes over Time).
      * In reality, Ketamine might drop depression scores incredibly fast for Subject 1, but very slowly for Subject 2, etc. 
      * We can add a ___Random Slope___ to explicitly model this variance
          * Changing our syntax to `(1 + Time | Subject)` tells the model: _"Give everyone their own starting baseline (intercept), AND allow everyone to have their own unique trajectory or rate of change (slope) over time."_
            
<img src="img/LMMs.png" width=600>
(<a href="https://peerj.com/articles/4794/">figure ref</a>)